# 🐍 Aula 17 — Automação de GPUs com Python

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

> 📌 **Relacionada à [Aula 14](../../aula14/README.md).** Aqui a automação é feita em **Python**,
> para rodar também no **Windows** e no **Colab** — sem depender de Bash.

---

## 🎯 Situação de aprendizagem

O servidor de treinamento da empresa travou de madrugada: a GPU atingiu **95 °C** e o job
falhou **sem avisar ninguém**. O time precisa de um sistema que, **sozinho**:
1. **colete** as métricas da GPU a cada poucos segundos;
2. **guarde** tudo num arquivo CSV (histórico);
3. **avise** quando a temperatura ou o uso passarem de um limite;
4. **mostre** um dashboard com os dados.

A diferença desta aula: faremos tudo em **Python**, que roda em qualquer sistema.

---

## 🧭 Como usar este notebook

1. Rode as células **na ordem** (de cima para baixo). Cada célula diz o que faz antes de rodar.
2. **Não precisa de GPU** para acompanhar: sem GPU NVIDIA, o notebook usa o **modo simulado**
   (cria valores parecidos com os de uma GPU real). Com GPU (Colab T4), os números são **reais**.
3. Onde aparecer `# ✏️ SUA VEZ`, é um **exercício** para você completar.

> 💡 **Ambiente:** foi feito para o **Google Colab** (Linux) e para o **Windows** do laboratório.
> Como tudo é Python puro, o mesmo código funciona nos dois.


## 1. Preparar o ambiente

Primeiro garantimos as bibliotecas que vamos usar. `pandas` e `matplotlib` normalmente já vêm
instaladas no Colab; `psutil` mede CPU e RAM.


In [ ]:
# @title 📦 Verificar/instalar dependencias
# ============================================================================
# OBJETIVO: garantir pandas, matplotlib e psutil. No Colab ja vem instaladas;
# em outros ambientes esta celula instala o que faltar.
# ============================================================================
import importlib
import subprocess
import sys

for pacote, modulo in [("pandas", "pandas"), ("matplotlib", "matplotlib"), ("psutil", "psutil")]:
    try:
        importlib.import_module(modulo)
        print(f"{pacote:<12} OK")
    except ImportError:
        print(f"{pacote:<12} instalando...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote])
print("\nDependencias prontas.")


## 2. Detectar a GPU (ou entrar em modo simulado)

Antes de coletar, precisamos saber **o que** temos: há uma GPU NVIDIA com `nvidia-smi`?
Se sim, lemos dados reais; se não, montamos um **simulador** com valores plausíveis — assim a
aula funciona em qualquer máquina.


In [ ]:
# @title 🔍 Detectar GPU e definir o modo (real ou simulado)
# ============================================================================
# OBJETIVO: decidir a fonte dos dados. shutil.which() procura o executavel no
# PATH; se achar nvidia-smi, TEM_GPU = True.
# ============================================================================
import shutil

TEM_GPU = shutil.which("nvidia-smi") is not None
print(f"nvidia-smi encontrado? {'SIM (dados reais)' if TEM_GPU else 'NAO (modo simulado)'}")

if TEM_GPU:
    import subprocess
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True).stdout)
else:
    print("Sem GPU NVIDIA: usaremos valores simulados para demonstrar o fluxo.")


## 3. Ler as métricas da GPU

Agora escrevemos a função `ler_gpu()` que devolve **um dicionário** com as métricas de **uma
amostra**. É o coração da automação: em Python, trocamos o `nvidia-smi` por um `dict`, bem mais
fácil de manipular.

> 📄 A mesma função existe no laboratório (`laboratorio_windows/1_monitor_gpu.py`), junto com a
> leitura real de **CPU e RAM** via `psutil`.


In [ ]:
# @title 📊 ler_gpu(): uma amostra de metricas (real ou simulada)
# ============================================================================
# OBJETIVO: devolver um dicionario com as metricas de UMA leitura da GPU.
# Com GPU NVIDIA, consulta o nvidia-smi; sem GPU, gera um valor plausivel.
# ============================================================================
import random
import subprocess

def ler_gpu():
    """Devolve dict com temperatura, uso, memoria e potencia da GPU."""
    if TEM_GPU:
        campos = ("index,name,temperature.gpu,utilization.gpu,utilization.memory,"
                  "memory.used,memory.total,power.draw,power.limit")
        saida = subprocess.check_output(
            ["nvidia-smi", f"--query-gpu={campos}", "--format=csv,noheader,nounits"],
            text=True).strip().split("\n")[0]
        idx, nome, temp, util, util_mem, usado, total, pot, limite = [
            c.strip() for c in saida.split(",")]
        return {"gpu_index": int(idx), "gpu_name": nome,
                "temp_c": float(temp), "util_gpu_pct": float(util),
                "util_mem_pct": float(util_mem), "mem_used_mb": float(usado),
                "mem_total_mb": float(total), "power_w": float(pot),
                "power_limit_w": float(limite)}
    # Modo simulado: valores parecidos com uma Tesla T4 em treino
    util = random.uniform(20, 95)
    return {"gpu_index": 0, "gpu_name": "Tesla T4 (simulada)",
            "temp_c": round(45 + util * 0.45, 1), "util_gpu_pct": round(util, 1),
            "util_mem_pct": round(util * 0.6, 1),
            "mem_used_mb": round(2048 + util * 40, 0), "mem_total_mb": 15360,
            "power_w": round(45 + util * 0.25, 1), "power_limit_w": 70.0}

# Teste rapido: mostre uma amostra
amostra = ler_gpu()
for chave, valor in amostra.items():
    print(f"  {chave:<14}: {valor}")


## 4. Coletar um histórico em CSV

Uma leitura só não conta a história. O coletor faz um **loop**: a cada `intervalo` segundos,
lê a GPU e acrescenta uma linha ao CSV. O cabeçalho é escrito **uma vez**; depois só anexamos.

> 🔎 **Como ler esta célula:** `csv.DictWriter` transforma o dicionário em uma linha do CSV;
> `f.flush()` garante que a linha seja salva imediatamente (importante em automação).


In [ ]:
# @title 🔁 Coletar metricas por alguns segundos -> gpu_log.csv
# ============================================================================
# OBJETIVO: rodar um loop de coleta e salvar cada amostra numa linha do CSV.
# Ajuste INTERVALO e AMOSTRAS para controlar a duracao (amostras * intervalo).
# ============================================================================
import csv
import time

ARQUIVO = "gpu_log.csv"
INTERVALO = 2      # segundos entre as amostras
AMOSTRAS = 5       # numero de amostras (5 x 2s = 10s de coleta)

CAMPOS = ["timestamp", "gpu_index", "gpu_name", "temp_c", "util_gpu_pct",
          "util_mem_pct", "mem_used_mb", "mem_total_mb", "power_w", "power_limit_w"]

print(f"Coletando {AMOSTRAS} amostras a cada {INTERVALO}s -> {ARQUIVO}\n")

# Abre o arquivo: escreve o cabecalho e acrescenta cada linha
with open(ARQUIVO, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=CAMPOS)
    escritor.writeheader()
    for i in range(1, AMOSTRAS + 1):
        linha = ler_gpu()
        linha["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")
        escritor.writerow(linha)
        f.flush()
        print(f"  [{i}/{AMOSTRAS}] temp={linha['temp_c']}C | "
              f"util={linha['util_gpu_pct']}% | mem={linha['mem_used_mb']:.0f}MB | "
              f"pot={linha['power_w']}W")
        if i < AMOSTRAS:
            time.sleep(INTERVALO)

print(f"\nColeta concluida. Historico salvo em '{ARQUIVO}'.")


## 5. Alertar quando passar do limite

Monitorar sem avisar não resolve o problema da madrugada. Aqui lemos o CSV e **registramos um
alerta** sempre que a temperatura ou o uso ultrapassarem um limite. É a versão Python do
`alerta_gpu.sh` da Aula 14.

> 🔎 **Como ler esta célula:** o limite padrão é `LIMITE_TEMP`; cada linha que passa vira uma
> mensagem no arquivo `gpu_alertas.log`.


In [ ]:
# @title 🚨 Verificar limites e registrar alertas
# ============================================================================
# OBJETIVO: ler o CSV e registrar um alerta quando temp/uso passarem do limite.
# ============================================================================
LIMITE_TEMP = 70    # graus Celsius
LIMITE_UTIL = 90    # porcentagem de uso
LOG = "gpu_alertas.log"

alertas = 0
with open(ARQUIVO, encoding="utf-8") as f, open(LOG, "w", encoding="utf-8") as log:
    for linha in csv.DictReader(f):
        temp = float(linha["temp_c"])
        util = float(linha["util_gpu_pct"])
        if temp >= LIMITE_TEMP:
            msg = f"[{linha['timestamp']}] ALERTA: temperatura {temp}C >= {LIMITE_TEMP}C"
            print(msg); log.write(msg + "\n"); alertas += 1
        if util >= LIMITE_UTIL:
            msg = f"[{linha['timestamp']}] ALERTA: utilizacao {util}% >= {LIMITE_UTIL}%"
            print(msg); log.write(msg + "\n"); alertas += 1

print(f"\n{alertas} alerta(s) registrado(s) em '{LOG}'.")


## 6. Simular o agendamento (o "cron" em Python)

Em servidor Linux, o `cron` roda o coletor a cada X minutos **automaticamente**. No Colab e no
Windows não há `cron`, então **simulamos** o agendamento com um loop Python: cada "ciclo" é como
se o cron tivesse disparado a tarefa naquele horário.

> 📄 Em produção (Linux), isso vira uma linha do crontab — veja a Aula 14.


In [ ]:
# @title ⏰ Simular 3 ciclos de coleta automatica (como o cron)
# ============================================================================
# OBJETIVO: repetir a coleta algumas vezes automaticamente, demonstrando o
# conceito de agendamento sem depender do cron do Linux.
# ============================================================================
CICLOS = 3

for ciclo in range(1, CICLOS + 1):
    print(f"--- Ciclo {ciclo}/{CICLOS} do agendador ---")
    amostra = ler_gpu()
    print(f"    temp={amostra['temp_c']}C | util={amostra['util_gpu_pct']}% "
          f"| pot={amostra['power_w']}W")
    if amostra["temp_c"] >= LIMITE_TEMP:
        print("    -> acima do limite! (em producao, dispararia um alerta)")
    time.sleep(1)  # em producao, aqui seriam minutos entre execucoes

print("\nSimulacao de agendamento concluida.")


## 7. Gerar o dashboard

Por fim, transformamos o CSV em **gráficos**. Não usamos `gnuplot` (do Linux): usamos
`pandas` + `matplotlib`, que funcionam igual no Colab e no Windows.


In [ ]:
# @title 📊 Dashboard: 4 graficos a partir do CSV
# ============================================================================
# OBJETIVO: ler o CSV e desenhar temperatura, uso, memoria e potencia.
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(ARQUIVO)
x = range(len(df))

fig, eixos = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Dashboard de Monitoramento da GPU", fontsize=14, fontweight="bold")

eixos[0, 0].plot(x, df["temp_c"], "o-", color="#EF4444")
eixos[0, 0].axhline(LIMITE_TEMP, color="#F97316", ls="--", label=f"limite {LIMITE_TEMP}C")
eixos[0, 0].set_title("Temperatura (C)"); eixos[0, 0].legend()

eixos[0, 1].plot(x, df["util_gpu_pct"], "s-", color="#10B981", label="GPU %")
eixos[0, 1].plot(x, df["util_mem_pct"], "^-", color="#6366F1", label="Mem %")
eixos[0, 1].set_title("Utilizacao (%)"); eixos[0, 1].legend()

eixos[1, 0].plot(x, df["mem_used_mb"], "o-", color="#7E22CE")
eixos[1, 0].set_title("Memoria usada (MB)")

eixos[1, 1].plot(x, df["power_w"], "o-", color="#F97316", label="consumo")
eixos[1, 1].plot(x, df["power_limit_w"], "--", color="#EF4444", label="limite")
eixos[1, 1].set_title("Potencia (W)"); eixos[1, 1].legend()

for eixo in eixos.flat:
    eixo.grid(True, ls=":", alpha=0.5)
plt.tight_layout()
plt.savefig("gpu_dashboard.png", dpi=120)
plt.show()
print("Dashboard gerado: gpu_dashboard.png")


## 8. (Opcional) Publicar no Google Sheets

Em produção, podemos enviar o CSV para uma planilha, para acompanhar de qualquer lugar. Isso usa
uma **Service Account** (credenciais). Aqui só **preparamos** os dados: sem a variável
`SHEETS_ID`, o script apenas conta as linhas que seriam enviadas.

> 📄 Passo a passo completo em [`docs/08_tutorial-google-sheets-api-gpu.md`](../../../../docs/08_tutorial-google-sheets-api-gpu.md).


In [ ]:
# @title 📤 Preparar envio ao Google Sheets (modo simulacao)
# ============================================================================
# OBJETIVO: mostrar como funciona o envio. Sem credenciais, apenas simula.
# ============================================================================
import os

sheets_id = os.environ.get("SHEETS_ID", "")
with open(ARQUIVO, encoding="utf-8") as f:
    total = sum(1 for _ in f) - 1  # descontar o cabecalho

if sheets_id:
    print(f"Enviando {total} registros para a planilha {sheets_id}... (requer credenciais)")
else:
    print(f"[Modo simulacao] SHEETS_ID nao definido.")
    print(f"Pronto para enviar {total} registros de telemetria ao Google Sheets.")


---

## 📝 Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Mudar a coleta.** Altere `INTERVALO` e `AMOSTRAS` para coletar **a cada 3 s durante 15 s**.
Quantas amostras o CSV terá?

**2) Novo limite.** Além da temperatura, adicione um alerta quando a **memória usada**
(`mem_used_mb`) passar de **8000 MB**.

**3) Nova métrica.** Acrescente `fan.speed` (rotação do cooler) na função `ler_gpu()` e no CSV.
Dica: adicione o campo na query do `nvidia-smi` e na lista `CAMPOS`.

**4) Gráfico novo.** Crie um gráfico que compare **temperatura × clock** ao longo do tempo, com
dois eixos Y.

**5) Webhook.** Escreva um trecho Python que monte o **JSON** que seria enviado a um webhook do
Slack/Discord quando houver alerta (não precisa enviar de verdade — só exibir o payload).

---

### Célula-esqueleto (complete para o exercício 4)

```python
# TODO: gráfico de temperatura x clock
# df = pd.read_csv(ARQUIVO)
# fig, ax1 = plt.subplots()
# ax1.plot(df["temp_c"], color="#EF4444", label="temp")
# ax2 = ax1.twinx()
# ...
```


---

## 📌 Síntese

- **Python + `subprocess`** permite consultar o `nvidia-smi` de dentro do programa.
- **CSV incremental** guarda o histórico — a base de qualquer telemetria.
- **Alertas por limite** transformam o monitor passivo em um sistema que avisa.
- **`pandas` + `matplotlib`** fazem o dashboard sem depender de ferramentas do Linux.
- **Agendamento** em Python (loop) funciona onde o `cron` não existe (Colab/Windows).

> 🔗 Próxima (Aula 18+19): operar a GPU com **concorrência** (lock/fila) e **energia**
> (Power Limit e eficiência).
